# Python to Java Code Converter


Convert Python code to Java using LLMs. Run and compare outputs and speed.

In [ ]:
# Imports

import os
import io
import sys
import time
import tempfile
import subprocess
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:8]}...")
else:
    print("OpenRouter API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}...")
else:
    print("Google API Key not set")

In [ ]:
# Connect to client libraries

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
openrouter_url = "https://openrouter.ai/api/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)

GEMINI_PRO = "gemini-2.5-pro"
GEMINI_FLASH = "gemini-2.5-flash"
OPENROUTER_GEMINI = "google/gemini-2.0-flash-001"
OPENROUTER_GPT = "openai/gpt-4o-mini"
OPENROUTER_CLAUDE = "anthropic/claude-3.5-sonnet"
OPENROUTER_LLAMA = "meta-llama/llama-3.1-70b-instruct"

models = [GEMINI_PRO, GEMINI_FLASH, OPENROUTER_GEMINI, OPENROUTER_GPT, OPENROUTER_CLAUDE, OPENROUTER_LLAMA]
DEFAULT_MODEL = GEMINI_FLASH  # use Flash by default (faster, less likely to timeout)
clients = {
    GEMINI_PRO: gemini,
    GEMINI_FLASH: gemini,
    OPENROUTER_GEMINI: openrouter,
    OPENROUTER_GPT: openrouter,
    OPENROUTER_CLAUDE: openrouter,
    OPENROUTER_LLAMA: openrouter,
}

In [ ]:
# System info 

week4_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, week4_path)

try:
    from system_info import retrieve_system_info
    system_info = retrieve_system_info()
except ImportError:
    import platform
    system_info = {
        "os": {"system": platform.system(), "arch": platform.machine()},
        "cpu": {"cores_logical": os.cpu_count() or 4, "brand": ""},
    }

print("System info loaded:", system_info.get('os', {}).get('system', 'N/A'))

In [ ]:
# Core conversion logic: system prompt and user prompt (Python → Java)

system_prompt = """Your task is to convert Python code into equivalent, idiomatic Java code.
Respond only with Java source code. Do not include explanations outside of brief comments in the code.
The Java code must:
1. Be complete and runnable (include a public class named Main with a main(String[] args) method if the Python script has top-level execution)
2. Preserve the behavior and output of the original Python (same logic, same print/output)
3. Use standard Java style: proper types, meaningful names, and common Java libraries (e.g., java.util, java.io) where appropriate
4. Map Python constructs to Java equivalents (list -> ArrayList, dict -> HashMap, print -> System.out.println, etc.)"""

def user_prompt_for(python_code):
    return f"""Convert this Python code to equivalent Java code with the same behavior and output.
System (for context): {system_info}

Original Python code:

```python
{python_code}
```

Respond only with the Java code. No explanations. Use ```java ... ``` for the code block. The public class must be named Main so it can be run."""

def messages_for(python_code):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python_code)}
    ]

In [ ]:
# LLM conversion function 

def convert_to_java(model, python_code):
    client = clients.get(model)
    if not client:
        return "Error: Model not available"
    response = client.chat.completions.create(
        model=model, messages=messages_for(python_code), timeout=180
    )
    reply = response.choices[0].message.content
    # Extract Java from markdown code block if present
    if "```java" in reply:
        reply = reply.split("```java")[1].split("```")[0].strip()
    elif "```" in reply:
        reply = reply.split("```")[1].split("```")[0].strip()
    return reply.strip()

In [ ]:
# Run Python and capture output 

def run_python(code):
    globals_dict = {"__builtins__": __builtins__}
    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer
    try:
        start = time.perf_counter()
        exec(code, globals_dict)
        elapsed = time.perf_counter() - start
        output = buffer.getvalue()
        return output.strip(), elapsed
    except Exception as e:
        sys.stdout = old_stdout
        return f"Error: {e}", -1.0
    finally:
        sys.stdout = old_stdout

In [ ]:
# Run Java: write to temp file, compile, run, capture output and time


def run_java(java_code):
    with tempfile.TemporaryDirectory() as tmpdir:
        java_path = os.path.join(tmpdir, "Main.java")
        with open(java_path, "w", encoding="utf-8") as f:
            f.write(java_code)
        try:
            comp = subprocess.run(
                ["javac", "-source", "8", "-target", "8", java_path],
                capture_output=True,
                text=True,
                timeout=30,
                cwd=tmpdir,
            )
            if comp.returncode != 0:
                return (comp.stderr or comp.stdout or "Compilation failed").strip(), -1.0
            # Find the class name (public class X)
            class_name = "Main"
            for line in java_code.splitlines():
                if "public class " in line:
                    class_name = line.split("public class ")[1].split()[0].strip()
                    break
            start = time.perf_counter()
            run_result = subprocess.run(
                ["java", class_name],
                capture_output=True,
                text=True,
                timeout=10,
                cwd=tmpdir,
            )
            elapsed = time.perf_counter() - start
            out = (run_result.stdout or "").strip()
            err = (run_result.stderr or "").strip()
            if run_result.returncode != 0:
                return (err or out or "Runtime error"), -1.0
            return out or "(no output)", elapsed
        except FileNotFoundError:
            return "Java not found (install JDK and add javac/java to PATH)", -1.0
        except subprocess.TimeoutExpired:
            return "Timeout", -1.0
        except Exception as e:
            return f"Error: {e}", -1.0

In [ ]:
# Example Python code to convert to Java


example_python = """
# Simple script: greet and sum a list
name = "World"
print(f"Hello, {name}!")


numbers = [10, 20, 30, 40, 50]
total = sum(numbers)
print(f"Sum of numbers: {total}")


# Heavy computation: sum of all primes up to 100,000
def is_prime(n):
    if n < 2:
        return False
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False
    return True


prime_total = 0
for i in range(2, 100001):
    if is_prime(i):
        prime_total += i


print(f"Sum of primes up to 100,000: {prime_total}")
"""

In [ ]:
# Styling for Gradio

CSS = """
:root {
  --py-color: #209dd7;
  --java-color: #ed8b00;
  --accent: #753991;
  --card: #161a22;
  --text: #e9eef5;
}
.gradio-container { max-width: 100% !important; padding: 0 40px !important; }
.convert-btn button { background: var(--accent) !important; color: white !important; font-weight: 700; }
.py-out textarea { background: rgba(32,157,215,.15); border: 1px solid rgba(32,157,215,.35); color: var(--py-color); font-weight: 600; }
.java-out textarea { background: rgba(237,139,0,.15); border: 1px solid rgba(237,139,0,.35); color: var(--java-color); font-weight: 600; }
"""

In [ ]:
# Gradio UI: convert Python to Java and optionally run both

def convert_and_benchmark(model, python_input):
    java_code = convert_to_java(model, python_input)
    orig_out, orig_time = run_python(python_input)
    java_out, java_time = run_java(java_code)

    orig_summary = f"Output: {orig_out}\nTime: {orig_time:.4f}s" if orig_time >= 0 else orig_out
    java_summary = f"Output: {java_out}\nTime: {java_time:.4f}s" if java_time >= 0 else java_out

    if orig_time > 0 and java_time > 0:
        ratio = orig_time / java_time
        java_summary += f"\nPython/Java time ratio: {ratio:.2f}x"

    return java_code, orig_summary, java_summary

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title="Python to Java Converter") as ui:
    gr.Markdown("## Python to Java Code Converter")
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_input = gr.Code(
                label="Original Python",
                value=example_python.strip(),
                language="python",
                lines=20
            )
        with gr.Column(scale=6):
            java_output = gr.Code(
                label="Generated Java",
                value="",
                language="cpp",  # java not in gr.Code.languages; cpp is close
                lines=20
            )
    with gr.Row():
        model = gr.Dropdown(models, value=DEFAULT_MODEL, label="Model")
        convert_btn = gr.Button("Convert to Java & Run", elem_classes=["convert-btn"])
    with gr.Row():
        with gr.Column(scale=6):
            orig_output = gr.TextArea(label="Python run result", lines=5, elem_classes=["py-out"])
        with gr.Column(scale=6):
            java_run_output = gr.TextArea(label="Java run result", lines=5, elem_classes=["java-out"])
    convert_btn.click(
        fn=convert_and_benchmark,
        inputs=[model, python_input],
        outputs=[java_output, orig_output, java_run_output]
    )

ui.launch()

## Summary


- Converts Python to Java with LLMs


- Supports Gemini, GPT, Claude, Llama


- Runs and compares both codes' output and speed


- Simple Gradio UI; Java runs if JDK is on PATH